# 標準住宅の外皮性能

床断熱・基礎断熱の等級4～7について、面積と層構成から UA・ηAC・ηAH を計算します。気象とソルバーは不要です。窓の日射熱取得率、等級4の開口部U値、土間の外周長・線熱貫流率は設計資料から記入してください。

In [ ]:
%pip -q install git+https://github.com/iguchi-lab/vtsimnx.git@v1.10.0

In [ ]:
import json
from pathlib import Path
import pandas as pd
import vtsimnx as vt

model = "foundation5"  # 床断熱は default, grade5～grade7。基礎断熱は foundation4～foundation7。
region = 6
house = vt.standard_house
types, layers = house.surface_types[model], house.layers[model]
input_data = {"surfaces": [
    {**row, "layers": layers.get(types[row["surface"]].get("layer_key"))}
    for row in house.surface_rows[model]
]}
# 室温計算用の input_data を組み立てた場合は、その辞書を直接使用できる。
path = Path(f"{model}_region{region}_rating.json")
if not path.exists():
    assessment = house.rating_template(model, region=region)
    path.write_text(json.dumps(assessment, ensure_ascii=False, indent=2), encoding="utf-8")
print(path.resolve())

左側のファイル一覧から表示されたJSONを開き、`null` に設計値を入力して保存します。窓の `eta_d` は枠込みの垂直面日射熱取得率（0～1）です。`lines` には隣接条件ごとの土間外周長 [m]、ψ [W/(m·K)]、温度差係数 H を記入します。編集済みJSONはダウンロードして保存できます。

In [ ]:
assessment = json.loads(path.read_text(encoding="utf-8"))
missing = []
for part in assessment["parts"]:
    name = part["keys"][0]
    if part["kind"] in {"window", "door"} and part.get("U") is None:
        missing.append((name, "U", "開口部全体の熱貫流率 [W/(m²·K)]"))
    if part["kind"] == "window":
        for season, label in [("C", "冷房期"), ("H", "暖房期")]:
            if season == "H" and assessment["region"] == 8:
                continue
            if part.get("eta_d_" + season, part.get("eta_d")) is None:
                missing.append((name, "eta_d / eta_d_" + season, label + "の枠込み日射熱取得率 [0～1]"))
for line in assessment["lines"]:
    for field, label in [("length", "長さ [m]"), ("psi", "線熱貫流率 [W/(m·K)]"), ("H", "温度差係数")]:
        if line.get(field) is None:
            missing.append((line["name"], field, label))

if missing:
    print(f"{path.name} に以下の設計値を入力して保存し、このセルを再実行してください。")
    print("窓の冷暖房期に同じ値を使う場合は eta_d だけを入力します。")
    display(pd.DataFrame(missing, columns=["部位", "JSONの項目", "必要な設計値"]))
else:
    result = vt.envelope_performance(input_data, assessment)
    display(pd.DataFrame([{k: v for k, v in result.items() if k != "parts"}]))
    display(pd.DataFrame(result["parts"]))